# Exercise 1: Integrating Data & Model Forms into the Governance Pipeline

This notebook digitizes the **Dataset Governance Form** and **Model Documentation Form**, then wires them into a NIST AI RMF (Govern → Map → Measure → Manage) gate for PayTrust AI fraud detection.


In [1]:
import pandas as pd
import numpy as np
from datetime import date
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 120)


## 1. Digitized Dataset Governance Form

Fields mirror `Governance Forms/Dataset_Governance_Form.pdf`.


In [2]:
data_form = pd.DataFrame([
    ("Dataset Name", "PayTrust Transaction Fraud Sample"),
    ("Dataset ID / Reference", "PT-DS-FRAUD-001"),
    ("Description", "Digital payment transactions with fraud labels and behavioural risk scores"),
    ("Business Purpose", "Train and validate real-time fraud scoring for PayTrust payments in Pakistan"),
    ("Source System / Origin", "Online public fraud-detection dataset (CSV) registered for lab use"),
    ("Data Format / Location", "CSV / Dataset/financial_fraud_detection_dataset.csv"),
    ("Business Owner", "Fraud Product Lead / Payments"),
    ("Data Steward", "AI Governance Officer"),
    ("Technical Owner", "ML Engineering"),
    ("Responsible Team", "AI Quality and Risk"),
    ("Classification", "Confidential"),
    ("Personal / Sensitive Data", "Yes - account identifiers, IP, device hash (restricted; dropped for modelling)"),
    ("Regulatory / Contractual Requirements", "Pakistan National AI Policy; payment-security internal policy"),
    ("Expected Quality Level", "High"),
    ("Key Quality Checks", "Completeness; Accuracy; Validity; Timeliness; Uniqueness"),
    ("Known Data Issues / Risks", "Severe class imbalance; missing time_since_last_transaction; fraud_type leakage if used as feature"),
    ("Approved Users / Roles", "AI governance committee; ML eng; QA (read-only)"),
    ("Access Method", "Local/CSV via approved notebook pipeline"),
    ("Access Approval", "AI Governance Officer"),
    ("Permitted Use", "Fraud model development, governance evaluation, audit evidence"),
    ("Update / Refresh Frequency", "Ad hoc (lab snapshot)"),
    ("Retention Period", "As required by lab / 2 years instructional copy"),
    ("Archival / Disposal", "Secure delete after course retention window"),
    ("Governance Status", "Under Review"),
    ("Approved By", "Pending gate decision in this notebook"),
    ("Approval Date", str(date.today())),
    ("Next Review Date", "2026-12-07"),
], columns=["Field", "Value"])

display(data_form)


,Field,Value
0,Dataset Name,PayTrust Transaction Fraud Sample
1,Dataset ID / Reference,PT-DS-FRAUD-001
2,Description,Digital payment transactions with fraud labels and behavioural risk scores
3,Business Purpose,Train and validate real-time fraud scoring for PayTrust payments in Pakistan
4,Source System / Origin,Online public fraud-detection dataset (CSV) registered for lab use
5,Data Format / Location,CSV / Dataset/financial_fraud_detection_dataset.csv
6,Business Owner,Fraud Product Lead / Payments
7,Data Steward,AI Governance Officer
8,Technical Owner,ML Engineering
9,Responsible Team,AI Quality and Risk


## 2. Digitized Model Documentation Form

Key fields adapted from `Governance Forms/Model_Documentation_Form.pdf` for the PayTrust fraud classifier.


In [3]:
model_form = pd.DataFrame([
    ("Document version", "1.0"),
    ("Last updated", str(date.today())),
    ("Legal name for the model provider", "PayTrust AI (instructional case)"),
    ("Model name", "PayTrust-Fraud-HGB-v1"),
    ("Model authenticity", "Notebook-trained artefact; hash recorded at export time"),
    ("Release date", "Not released — pending governance gate"),
    ("License", "Internal / instructional use only"),
    ("Acceptable Use Policy", "Fraud scoring for payments only; no automated loan decisions from this model"),
    ("Intended uses", "Real-time fraud risk scoring of digital payments; human review for uncertain scores"),
    ("Prohibited uses", "Credit scoring; identity verification as sole evidence; automated irreversible blocks without appeal"),
    ("Approved features", "amount, transaction_type, merchant_category, location, device_used, payment_channel, time_since_last_transaction, spending_deviation_score, velocity_score, geo_anomaly_score, hour, day_of_week, month"),
    ("Excluded / leakage features", "transaction_id, sender_account, receiver_account, ip_address, device_hash, fraud_type"),
    ("Training / validation / test split", "Stratified 60/20/20 on sampled balanced-aware set"),
    ("Primary metrics", "Precision, Recall, F1, ROC-AUC, False Positive Rate, expected cost"),
    ("Known limitations", "Imbalanced labels; online data may not match live Pakistani traffic; segment fairness must be monitored"),
    ("Human oversight", "Review queue for mid-score band; customer appeal path required before full launch"),
    ("Monitoring and rollback", "Weekly FPR/recall drift checks; rollback kill-switch to shadow mode"),
    ("Model owner", "Fraud Product Lead"),
    ("Governance Status", "Under Review"),
], columns=["Field", "Value"])

display(model_form)


,Field,Value
0,Document version,1.0
1,Last updated,2026-11-07
2,Legal name for the model provider,PayTrust AI (instructional case)
3,Model name,PayTrust-Fraud-HGB-v1
4,Model authenticity,Notebook-trained artefact; hash recorded at export time
5,Release date,Not released — pending governance gate
6,License,Internal / instructional use only
7,Acceptable Use Policy,Fraud scoring for payments only; no automated loan decisions from this model
8,Intended uses,Real-time fraud risk scoring of digital payments; human review for uncertain scores
9,Prohibited uses,Credit scoring; identity verification as sole evidence; automated irreversible blocks without appeal


## 3. NIST pipeline gates that consume the forms

Govern uses ownership fields. Map uses purpose, sensitivity, excluded features. Measure uses metric thresholds. Manage uses oversight and rollback fields.


In [4]:
def field_value(df, field):
    rows = df.loc[df["Field"] == field, "Value"]
    return str(rows.iloc[0]) if len(rows) else ""

REQUIRED_DATA_FIELDS = [
    "Dataset Name", "Business Purpose", "Business Owner", "Data Steward",
    "Classification", "Personal / Sensitive Data", "Permitted Use", "Governance Status",
]
REQUIRED_MODEL_FIELDS = [
    "Model name", "Intended uses", "Prohibited uses", "Approved features",
    "Excluded / leakage features", "Primary metrics", "Human oversight",
    "Monitoring and rollback", "Model owner", "Governance Status",
]

def form_completeness(df, required):
    checks = []
    for f in required:
        val = field_value(df, f).strip()
        ok = bool(val) and val.lower() not in {"", "tbd", "n/a", "pending"}
        checks.append({"Form field": f, "Complete": ok, "Value": val[:80]})
    return pd.DataFrame(checks)

data_checks = form_completeness(data_form, REQUIRED_DATA_FIELDS)
model_checks = form_completeness(model_form, REQUIRED_MODEL_FIELDS)
display(Markdown("### Dataset form completeness"))
display(data_checks)
display(Markdown("### Model form completeness"))
display(model_checks)


### Dataset form completeness

,Form field,Complete,Value
0,Dataset Name,True,PayTrust Transaction Fraud Sample
1,Business Purpose,True,Train and validate real-time fraud scoring for PayTrust payments in Pakistan
2,Business Owner,True,Fraud Product Lead / Payments
3,Data Steward,True,AI Governance Officer
4,Classification,True,Confidential
5,Personal / Sensitive Data,True,"Yes - account identifiers, IP, device hash (restricted; dropped for modelling)"
6,Permitted Use,True,"Fraud model development, governance evaluation, audit evidence"
7,Governance Status,True,Under Review


### Model form completeness

,Form field,Complete,Value
0,Model name,True,PayTrust-Fraud-HGB-v1
1,Intended uses,True,Real-time fraud risk scoring of digital payments; human review for uncertain sco
2,Prohibited uses,True,Credit scoring; identity verification as sole evidence; automated irreversible b
3,Approved features,True,"amount, transaction_type, merchant_category, location, device_used, payment_chan"
4,Excluded / leakage features,True,"transaction_id, sender_account, receiver_account, ip_address, device_hash, fraud"
5,Primary metrics,True,"Precision, Recall, F1, ROC-AUC, False Positive Rate, expected cost"
6,Human oversight,True,Review queue for mid-score band; customer appeal path required before full launc
7,Monitoring and rollback,True,Weekly FPR/recall drift checks; rollback kill-switch to shadow mode
8,Model owner,True,Fraud Product Lead
9,Governance Status,True,Under Review


In [5]:
# Illustrative Measure metrics from an ungated accuracy-focused model
measure = {
    "accuracy": 0.963,
    "fraud_recall": 0.77,
    "fraud_precision": 0.36,
    "fpr": 0.033,
    "segment_fpr_gap": 0.04,
}

POLICY = {
    "max_fpr": 0.02,
    "min_recall": 0.70,
    "min_precision": 0.40,
    "max_segment_fpr_gap": 0.03,
    "require_human_oversight": True,
    "require_rollback": True,
}

nist_gates = pd.DataFrame([
    ("GOVERN", "Data + Model form ownership complete",
     bool(data_checks["Complete"].all() and model_checks["Complete"].all()),
     "All required form fields filled"),
    ("GOVERN", "Named Business Owner and Model owner",
     bool(field_value(data_form, "Business Owner") and field_value(model_form, "Model owner")),
     "Accountability on both forms"),
    ("MAP", "Intended vs prohibited uses documented",
     bool(field_value(model_form, "Intended uses") and field_value(model_form, "Prohibited uses")),
     "Use boundaries on Model form"),
    ("MAP", "Leakage / PII exclusions documented",
     "fraud_type" in field_value(model_form, "Excluded / leakage features").lower(),
     "Excluded features list includes fraud_type"),
    ("MAP", "Sensitive data acknowledged",
     "yes" in field_value(data_form, "Personal / Sensitive Data").lower(),
     "Dataset form sensitivity section"),
    ("MEASURE", f"FPR <= {POLICY['max_fpr']:.0%}",
     measure["fpr"] <= POLICY["max_fpr"],
     f"Observed FPR={measure['fpr']:.3f}"),
    ("MEASURE", f"Fraud recall >= {POLICY['min_recall']:.0%}",
     measure["fraud_recall"] >= POLICY["min_recall"],
     f"Observed recall={measure['fraud_recall']:.3f}"),
    ("MEASURE", f"Fraud precision >= {POLICY['min_precision']:.0%}",
     measure["fraud_precision"] >= POLICY["min_precision"],
     f"Observed precision={measure['fraud_precision']:.3f}"),
    ("MEASURE", f"Segment FPR gap <= {POLICY['max_segment_fpr_gap']:.0%}",
     measure["segment_fpr_gap"] <= POLICY["max_segment_fpr_gap"],
     f"Observed gap={measure['segment_fpr_gap']:.3f}"),
    ("MANAGE", "Human oversight documented",
     POLICY["require_human_oversight"] and "review" in field_value(model_form, "Human oversight").lower(),
     field_value(model_form, "Human oversight")[:60]),
    ("MANAGE", "Monitoring and rollback documented",
     POLICY["require_rollback"] and "rollback" in field_value(model_form, "Monitoring and rollback").lower(),
     field_value(model_form, "Monitoring and rollback")[:60]),
], columns=["NIST function", "Gate", "Pass", "Evidence"])

display(nist_gates)


,NIST function,Gate,Pass,Evidence
0,GOVERN,Data + Model form ownership complete,True,All required form fields filled
1,GOVERN,Named Business Owner and Model owner,True,Accountability on both forms
2,MAP,Intended vs prohibited uses documented,True,Use boundaries on Model form
3,MAP,Leakage / PII exclusions documented,True,Excluded features list includes fraud_type
4,MAP,Sensitive data acknowledged,True,Dataset form sensitivity section
5,MEASURE,FPR <= 2%,False,Observed FPR=0.033
6,MEASURE,Fraud recall >= 70%,True,Observed recall=0.770
7,MEASURE,Fraud precision >= 40%,False,Observed precision=0.360
8,MEASURE,Segment FPR gap <= 3%,False,Observed gap=0.040
9,MANAGE,Human oversight documented,True,Review queue for mid-score band; customer appeal path requir


## 4. Pipeline decision (forms + gates)


In [6]:
passed = int(nist_gates["Pass"].sum())
total = len(nist_gates)
failed = nist_gates.loc[~nist_gates["Pass"], ["NIST function", "Gate", "Evidence"]]

forms_ok = bool(data_checks["Complete"].all() and model_checks["Complete"].all())
measure_ok = bool(nist_gates.loc[nist_gates["NIST function"] == "MEASURE", "Pass"].all())
manage_ok = bool(nist_gates.loc[nist_gates["NIST function"] == "MANAGE", "Pass"].all())

if forms_ok and nist_gates["Pass"].all():
    decision = "GO"
    rationale = "Forms complete and all NIST gates passed. Controlled pilot allowed."
elif forms_ok and manage_ok and not measure_ok:
    decision = "CONDITIONAL / SHADOW MODE"
    rationale = "Forms and oversight exist, but Measure thresholds failed. Score in shadow mode; no auto-blocks."
else:
    decision = "NO-GO"
    rationale = "Critical form or governance gates failed. Halt unrestricted release."

data_form.loc[data_form["Field"] == "Governance Status", "Value"] = (
    "Approved" if decision == "GO" else "Under Review"
)
model_form.loc[model_form["Field"] == "Governance Status", "Value"] = (
    "Approved" if decision == "GO" else "Under Review"
)

display(Markdown(f'''### Decision: **{decision}**

Gates passed: **{passed}/{total}**

{rationale}
'''))
if len(failed):
    display(Markdown("#### Failed gates"))
    display(failed)

memo = f'''PAYTRUST AI — GOVERNANCE PIPELINE DECISION (EXERCISE 1)
Decision: {decision}
Date: {date.today()}
Forms integrated: Dataset Governance Form + Model Documentation Form
NIST gates passed: {passed}/{total}
Failed gates: {', '.join(failed['Gate'].tolist()) if len(failed) else 'None'}
Next step: {'Controlled pilot with monitoring' if decision == 'GO' else 'Remediate failed Measure/Manage items; keep model offline or shadow-only'}
'''
print(memo)


### Decision: **CONDITIONAL / SHADOW MODE**

Gates passed: **8/11**

Forms and oversight exist, but Measure thresholds failed. Score in shadow mode; no auto-blocks.


#### Failed gates

,NIST function,Gate,Evidence
5,MEASURE,FPR <= 2%,Observed FPR=0.033
7,MEASURE,Fraud precision >= 40%,Observed precision=0.360
8,MEASURE,Segment FPR gap <= 3%,Observed gap=0.040


PAYTRUST AI — GOVERNANCE PIPELINE DECISION (EXERCISE 1)
Decision: CONDITIONAL / SHADOW MODE
Date: 2026-11-07
Forms integrated: Dataset Governance Form + Model Documentation Form
NIST gates passed: 8/11
Failed gates: FPR <= 2%, Fraud precision >= 40%, Segment FPR gap <= 3%
Next step: Remediate failed Measure/Manage items; keep model offline or shadow-only

